In [5]:
from pydantic import BaseModel, ConfigDict

class Test(BaseModel):
    model_config = ConfigDict(extra="allow")

    send_progress: bool = True 
    send_tool_hints: bool = False


test = Test(send_progress=True, send_tool_hints=True, qq='123')

In [8]:
test.qq

'123'

In [15]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import AIMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location."""
    return f"The weather in {location} is 70 degrees and sunny."
    # raise ValueError("Error")

agent = create_agent(
    model=init_chat_model("deepseek-chat"),
    tools=[get_weather],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "get_weather": True
        }
    )],
    checkpointer=MemorySaver()
)

In [16]:
config={"configurable": {"thread_id": "1"}}
async for event in agent.astream_events({"messages": [{"role": "user", "content": "苏州今天天气如何啊?"}]},config=config):
    event_type = event.get("event", "")
    # if event_type == "on_chat_model_stream":
    #     continue
    # #     output: AIMessage = event["data"].get("output")
    # #     print(output.text)
    # print(event)
    # print('-----' * 10)

    if event_type == "on_chain_stream":
        # print(event)
        # print('--------' * 10)
        chunk = event['data'].get('chunk')
        # print(chunk)
        # print('--------' * 10)
        if isinstance(chunk, dict) and '__interrupt__' in chunk:

            interrupt_list = chunk['__interrupt__']
            for interrupt in interrupt_list:
                print(interrupt)
                print(interrupt.value)
            
            


Interrupt(value={'action_requests': [{'name': 'get_weather', 'args': {'location': '苏州'}, 'description': "Tool execution requires approval\n\nTool: get_weather\nArgs: {'location': '苏州'}"}], 'review_configs': [{'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='34af58e3d08a3e38bac40b24b3520be7')
{'action_requests': [{'name': 'get_weather', 'args': {'location': '苏州'}, 'description': "Tool execution requires approval\n\nTool: get_weather\nArgs: {'location': '苏州'}"}], 'review_configs': [{'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']}]}


In [17]:
from langgraph.types import Command
cmd = Command(
    resume={
        "decisions":[
            {
                "type": "approve"
            }
        ]
    }
)
async for event in agent.astream_events(cmd,config=config):
    event_type = event.get("event", "")
    if event_type == "on_chat_model_stream":
        continue
    #     output: AIMessage = event["data"].get("output")
    #     print(output.text)
    print(event)
    print('-----' * 10)

{'event': 'on_chain_start', 'data': {'input': Command(resume={'decisions': [{'type': 'approve'}]})}, 'name': 'LangGraph', 'tags': [], 'run_id': '019d14cd-867f-7a02-9bcc-6221751fb45e', 'metadata': {'thread_id': '1'}, 'parent_ids': []}
--------------------------------------------------
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='苏州今天天气如何啊?', additional_kwargs={}, response_metadata={}, id='fc44d368-b99a-493b-a234-30e860241615'), AIMessage(content='我来帮您查询苏州今天的天气情况。', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'model_provider': 'deepseek'}, id='lc_run--019d14cd-6b8d-7f81-bc0d-ae7a21950848', tool_calls=[{'name': 'get_weather', 'args': {'location': '苏州'}, 'id': 'call_00_ikTlTB41X97OddVrI0GYaCyS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 305, 'output_tokens': 52, 'total_tokens': 357, 'input